# 07 · Hyperparameter tuning — FashionMNIST MLP

Grid search, random search, Optuna. Subsets keep Colab runtime sane.

Source: `Model_Optimization_HyperparameterTuning_Exercise`

Course asks for 100 base epochs and 50 final epochs. That is slow on CPU. Defaults here: **8 / 3 / 12**. Bump `BASE_EPOCHS`, `SEARCH_EPOCHS`, `FINAL_EPOCHS` if you have a GPU.

In [ ]:
try:
    import optuna
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'])
    import optuna

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

BASE_EPOCHS = 8
SEARCH_EPOCHS = 3
FINAL_EPOCHS = 12

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_subset_size, test_subset_size = 4000, 1000
train_subset, _ = random_split(dataset, [train_subset_size, len(dataset) - train_subset_size],
                               generator=torch.Generator().manual_seed(0))
test_subset, _ = random_split(test_dataset, [test_subset_size, len(test_dataset) - test_subset_size],
                              generator=torch.Generator().manual_seed(0))

batch_size = 32
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)
print('Training data size:', len(train_subset))
print('Testing data size:', len(test_subset))

In [ ]:
class FashionNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.network(x)


def one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = correct = n = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += loss_fn(logits, labels).item()
            correct += (logits.argmax(1) == labels).sum().item()
            n += labels.size(0)
    return total_loss / len(loader), 100 * correct / n

### Base model — SGD lr=0.01, batch 32

In [ ]:
model = FashionNN().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)
base_hist = []
for epoch in range(BASE_EPOCHS):
    train_loss = one_epoch(model, train_loader, optimizer, loss_function)
    base_hist.append(train_loss)
    print(f'Epoch {epoch+1}/{BASE_EPOCHS}, Loss: {train_loss:.4f}')
print('base test', evaluate(model, test_loader, loss_function))

### Grid search — lr × batch size (short runs)

In [ ]:
learning_rates = [0.001, 0.01, 0.1]
batch_sizes = [32, 64]
best_loss = float('inf')
best_params = {}

for lr in learning_rates:
    for bs in batch_sizes:
        loader = DataLoader(train_subset, batch_size=bs, shuffle=True)
        net = FashionNN().to(device)
        opt = optim.SGD(net.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()
        last = None
        for _ in range(SEARCH_EPOCHS):
            last = one_epoch(net, loader, opt, loss_fn)
        print(f'LR: {lr}, Batch size: {bs}, Loss: {last:.4f}')
        if last < best_loss:
            best_loss = last
            best_params = {'lr': lr, 'batch_size': bs}

print('Best Params (Grid Search):', best_params)

### Random search — 5 trials

In [ ]:
learning_rates_r = [0.0001, 0.001, 0.01, 0.1]
batch_sizes_r = [16, 32, 64, 128]
random.seed(0)
best_rand = {'loss': float('inf')}

for trial in range(5):
    lr = random.choice(learning_rates_r)
    bs = random.choice(batch_sizes_r)
    loader = DataLoader(train_subset, batch_size=bs, shuffle=True)
    net = FashionNN().to(device)
    opt = optim.SGD(net.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    last = None
    for _ in range(SEARCH_EPOCHS):
        last = one_epoch(net, loader, opt, loss_fn)
    print(f'trial {trial+1}  LR: {lr}, Batch size: {bs}, Loss: {last:.4f}')
    if last < best_rand['loss']:
        best_rand = {'lr': lr, 'batch_size': bs, 'loss': last}
        if last < best_loss:
            best_loss = last
            best_params = {'lr': lr, 'batch_size': bs}

print('Best Params (Random Search):', best_rand)

### Optuna — lr and hidden width

In [ ]:
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-1, log=True)
    neurons = trial.suggest_int('neurons', 32, 256)
    net = nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, neurons),
        nn.ReLU(),
        nn.Linear(neurons, 10),
    ).to(device)
    opt = optim.SGD(net.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(SEARCH_EPOCHS):
        one_epoch(net, train_loader, opt, loss_fn)
    val_loss, _ = evaluate(net, test_loader, loss_fn)
    return val_loss

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=8)
print('Best Params (Optuna):', study.best_params)

### Retrain with best grid/random params

In [ ]:
print('using', best_params)
model = FashionNN().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=best_params['lr'])
final_loader = DataLoader(train_subset, batch_size=best_params['batch_size'], shuffle=True)
hist = []
for epoch in range(FINAL_EPOCHS):
    train_loss = one_epoch(model, final_loader, optimizer, loss_function)
    hist.append(train_loss)
    print(f'Epoch {epoch+1}/{FINAL_EPOCHS}, Loss: {train_loss:.4f}')

test_loss, test_acc = evaluate(model, test_loader, loss_function)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.2f}%')

plt.figure(figsize=(7, 3.5))
plt.plot(hist)
plt.title('Final model train loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
predictions = model(images.to(device)).argmax(dim=1).cpu()
plt.figure(figsize=(7, 7))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].squeeze(), cmap='gray')
    plt.title(f'Pred: {predictions[i].item()}, True: {labels[i].item()}')
    plt.axis('off')
plt.tight_layout()
plt.show()